# Libraries & Paths

In [30]:
import random, os, json, warnings
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.sparse as sp
from pathlib import Path
import decoupler as dc

# Seed of use for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Wandb import
#import wandb
#wandb.login()  
# Use of wandb (Weights & Biases) to track model configurations, performance metrics, plots, and experiment comparisons in a reproducible way.

# Setting the path of input & output data 
data_dir   = Path('../data')   
output_dir = Path('../outputs')
output_dir.mkdir(parents=True, exist_ok=True)
figures_dir = Path("../figures")
figures_dir.mkdir(parents=True, exist_ok=True)
filtered_anndata  = output_dir / "filtered_anndata.h5ad" 


warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
print ("Loaded")

Loaded


# Loading AnnData

In [2]:
# Loading Annadata
adata = sc.read_h5ad(filtered_anndata)


# Reloading the anndata for re-checking its values and shape while also printing the expression value range of the samples in the expression matrix.
print(f"AnnData shape: {adata.shape}")
print(f"obs columns: {list(adata.obs.columns)}")
print(f"First 5 gene names: {adata.var_names[:5].tolist()}")
X_sample = adata.X[:100, :100]
if sp.issparse(X_sample):
    X_sample = X_sample.toarray()
print(f"Expression value range of samples: [{X_sample.min():.2f}, {X_sample.max():.2f}]")

AnnData shape: (16291, 10019)
obs columns: ['sample', 'response_label', 'therapy', 'timepoint', 'patient_id', 'response', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt']
First 5 gene names: ['DPM1', 'SCYL3', 'C1orf112', 'FGR', 'FUCA2']
Expression value range of samples: [0.00, 12.24]


---
# Per-cell Pathway Activity Score

In [ ]:
# Loading PROGENy pathway-gene weight matrix where top = 500 (default) responsive genes per pathway are used.
progeny= dc.op.progeny(organism='human', top=500) # [9]
print(f"PROGENy dataset shape: {progeny.shape}")
print(f"Column info: {progeny.columns.tolist()}")
print(f"Total pathways: {progeny['source'].nunique()}")
print(f"Naming of Pathways: {sorted(progeny['source'].unique())}")

display(progeny.head(10))

PROGENy network shape: (6463, 4)
Columns: ['source', 'target', 'weight', 'padj']
Number of pathways: 14
Pathways: ['Androgen', 'EGFR', 'Estrogen', 'Hypoxia', 'JAK-STAT', 'MAPK', 'NFkB', 'PI3K', 'TGFb', 'TNFa', 'Trail', 'VEGF', 'WNT', 'p53']


,source,target,weight,padj
0,Androgen,TMPRSS2,11.490631,2.384806e-47
1,Androgen,NKX3-1,10.622551,2.205102e-44
2,Androgen,MBOAT2,10.472733,4.632376e-44
3,Androgen,KLK2,10.176186,1.944410e-40
4,Androgen,SARG,11.386852,2.790210e-40
5,Androgen,SLC38A4,7.363805,1.253071e-39
6,Androgen,MTMR9,6.130646,2.534403e-38
7,Androgen,ZBTB16,10.614437,1.567152e-36
8,Androgen,KCNN2,9.473199,7.711872e-36
9,Androgen,OPRK1,-5.626074,1.114245e-35


## Overlap check PROGENy \ AnnData

In [ ]:
progeny_genes   = set(progeny["target"].unique())
adata_genes     = set(adata.var_names.astype(str))
overlap_genes   = progeny_genes & adata_genes

print(f"PROGENy uses {len(progeny_genes):,}")
print(f"AnnData has {len(adata_genes):,} genes")
print(f"Overlap: {len(overlap_genes):,} genes ({len(overlap_genes)/len(progeny_genes)*100:.1f}% of PROGENy genes found)")

# Having >= 50% overlap is sufficient for reliable PROGENy scoring

PROGENy uses 5,276
AnnData has 10,019 genes
Overlap: 3,120 genes (59.1% of PROGENy genes found)


**Comment:** Of the 10,019 genes remaining after filtering, 3,120 overlap with the PROGENy gene set, representing 59.1% of the PROGENy genes identified, which are used to calculate pathway activity scores.   

Those pathway scores will be added as additional features, possibly providing biological information about the activity of major signaling pathways that may not be directly captured from individual gene-expression values alone.

---

# References
 
> [9].Package ‘decoupleR’: https://www.bioconductor.org/packages//release/bioc/manuals/decoupleR/man/decoupleR.pdf <\br>
 